In [1]:
import pandas as pd
import os

In [2]:
RAW_PATH = "../datasets/raw/"
PROCESSED_PATH = "../datasets/processed/"

In [3]:
os.listdir(RAW_PATH)

['completion_rate_2023.csv',
 'education_25plus_2025.csv',
 'eys_2025.csv',
 'facilities_2025.csv',
 'libraries_2025.csv',
 'literacy_2025.csv',
 'reading_fondness_2025.csv',
 'schools_2025.csv']

In [17]:
schools = pd.read_csv(
    os.path.join(RAW_PATH, "schools_2025.csv"),
    header=0
)

schools.columns = [
    "province",
    "schools_primary_public",
    "schools_primary_private",
    "schools_primary_total",
    "teachers_primary_public",
    "teachers_primary_private",
    "teachers_primary_total",
    "pupils_primary_public",
    "pupils_primary_private",
    "pupils_primary_total"
]

In [18]:
schools.head()
schools.columns

Index(['province', 'schools_primary_public', 'schools_primary_private',
       'schools_primary_total', 'teachers_primary_public',
       'teachers_primary_private', 'teachers_primary_total',
       'pupils_primary_public', 'pupils_primary_private',
       'pupils_primary_total'],
      dtype='object')

In [19]:
schools.to_csv(
    os.path.join(PROCESSED_PATH, "schools_2025_processed.csv"),
    index=False
)

In [21]:
libraries = pd.read_csv(
    os.path.join(RAW_PATH, "libraries_2025.csv"),
    header=0
)

libraries.columns = [
    "province",
    "special_library_a",
    "special_library_b",
    "special_library_c",
    "special_library_total",
    "school_library_a",
    "school_library_b",
    "school_library_c",
    "school_library_total",
    "academic_library_a",
    "academic_library_b",
    "academic_library_c",
    "academic_library_total",
    "public_library_a",
    "public_library_b",
    "public_library_c",
    "public_library_total",
    "public_library_all"
]

In [22]:
libraries.columns = [
    "province",
    "special_library_a",
    "special_library_b",
    "special_library_c",
    "special_library_total",
    "school_library_a",
    "school_library_b",
    "school_library_c",
    "school_library_total",
    "academic_library_a",
    "academic_library_b",
    "academic_library_c",
    "academic_library_total",
    "public_library_a",
    "public_library_b",
    "public_library_c",
    "public_library_total",
    "public_library_all"
]

In [23]:
libraries.to_csv(
    os.path.join(PROCESSED_PATH, "libraries_2025_processed.csv"),
    index=False
)

In [24]:

reading_fondness = pd.read_csv(
    os.path.join(RAW_PATH, "reading_fondness_2025.csv"),
    header=0
)

reading_fondness.columns = [
    "province",
    "reading_fondness_level",
    "tgm_pre_reading",
    "tgm_reading",
    "tgm_post_reading"
]

In [25]:
reading_fondness.to_csv(
    os.path.join(PROCESSED_PATH, "reading_fondness_2025_processed.csv"),
    index=False
)

EYS (Data multi-level)
Row 0–1 = title
Row 2 = gender
Row 3 = year
Row 4 = data

In [10]:
eys = pd.read_csv(
    os.path.join(RAW_PATH, "eys_2025.csv"),
    header=[2, 3]
)

eys.columns = [
    "region",
    "eys_male",
    "eys_female"
]

In [26]:
eys.to_csv(
    os.path.join(PROCESSED_PATH, "eys_2025_processed.csv"),
    index=False
)

In [28]:
completion = pd.read_csv(
    os.path.join(RAW_PATH, "completion_rate_2023.csv"),
    header=[2, 3]
)

completion.columns = [
    "province",
    "completion_elementary",
    "completion_junior_high",
    "completion_senior_high"
]

In [ ]:
completion.to_csv(
    os.path.join(PROCESSED_PATH, "completion_2023_processed.csv"),
    index=False
)

In [30]:
facilities = pd.read_csv(
    os.path.join(RAW_PATH, "facilities_2025.csv"),
    header=[2, 3]
)

facilities.columns = [
    "province",
    "villages_primary",
    "villages_junior_high",
    "villages_senior_high",
    "villages_vocational",
    "villages_university"
]

In [31]:
facilities.to_csv(
    os.path.join(PROCESSED_PATH, "facilities_2025_processed.csv"),
    index=False
)

In [32]:
education_25plus = pd.read_csv(
    os.path.join(RAW_PATH, "education_25plus_2025.csv"),
    header=[2, 3]
)

education_25plus.columns = [
    "region",
    "education_25plus_male",
    "education_25plus_female"
]

In [33]:
education_25plus.to_csv(
    os.path.join(PROCESSED_PATH, "education_25plus_2025_processed.csv"),
    index=False
)

In [47]:
literacy = pd.read_csv(
    os.path.join(RAW_PATH, "literacy_2025.csv"),
    header=None,
    encoding="utf-8-sig",
)

In [48]:
# 2. The file is 3 stacked blocks (Urban / Rural / Urban+Rural), each with its
#    own "Province" header row. Find where each block starts.
block_starts = literacy.index[literacy[0] == "Province"].tolist()
 
blocks = []
for i, start in enumerate(block_starts):
    classification = literacy.iat[start, 2]          # "Urban" / "Rural" / "Urban+Rural"
    sex_row  = literacy.iloc[start + 1].ffill()       # Male / Female / Male+Female
    year_row = literacy.iloc[start + 2]               # 2009 ... 2025
 
    cols = ["Province", None]  # col0 = Province, col1 = empty classification col
    for s, y in zip(sex_row[2:], year_row[2:]):
        if str(y).strip().isdigit():
            cols.append(f"{s}_{y}")
        else:
            cols.append(None)                    # trailing junk column
 
    # Data rows run until the next block header (or end of file)
    end = block_starts[i + 1] if i + 1 < len(block_starts) else len(literacy)
    chunk = literacy.iloc[start + 3:end].copy()
    chunk.columns = cols[:len(chunk.columns)]
 
    chunk = chunk.loc[:, chunk.columns.notna()]
    chunk = chunk[chunk["Province"].notna() & (chunk["Province"].str.strip() != "")]
    chunk = chunk[~chunk["Province"].str.contains("Source", case=False, na=False)]
 
    chunk.insert(1, "Urban_Rural", classification)
    blocks.append(chunk)
 
data = pd.concat(blocks, ignore_index=True)

In [49]:
# 3. BPS marks missing values with "-" -> convert to real NaN
data = data.replace(r"^\s*-\s*$", pd.NA, regex=True)

In [50]:
# 4. Strip stray whitespace from text columns
for c in ("Province", "Urban_Rural"):
    data[c] = data[c].astype(str).str.strip()

In [51]:
# 5. Convert year columns to numeric
value_cols = [c for c in data.columns if c not in ("Province", "Urban_Rural")]
data[value_cols] = data[value_cols].apply(pd.to_numeric, errors="coerce")

In [52]:
# 6. Drop rows that are entirely missing across all value columns (true empty rows)
data = data.dropna(subset=value_cols, how="all").reset_index(drop=True)
 
print("shape:", data.shape)
print(data["Urban_Rural"].value_counts())
print(data.head())
print("remaining NaN cells (genuine missing data, e.g. new provinces):", data.isna().sum().sum())

shape: (116, 53)
Urban_Rural
Urban          39
Urban+Rural    39
Rural          38
Name: count, dtype: int64
         Province Urban_Rural  Male_2009  Male_2010  Male_2011  Male_2012  \
0            Aceh       Urban      99.34      99.08      98.82      99.08   
1  Sumatera Utara       Urban      99.33      99.34      99.12      99.56   
2  Sumatera Barat       Urban      99.39      98.88      98.51      98.90   
3            Riau       Urban      99.34      99.44      99.27      99.22   
4           Jambi       Urban      99.11      98.47      98.86      99.12   

   Male_2013  Male_2014  Male_2015  Male_2016  ...  Male+Female_2016  \
0      99.37      99.22      99.40      99.64  ...             99.04   
1      99.56      99.91      99.80      99.84  ...             99.54   
2      99.27      99.71      99.80      99.58  ...             99.29   
3      99.34      99.72      99.88      99.88  ...             99.59   
4      99.03      99.39      99.48      99.18  ...             98.98

In [53]:
# 7 save data to csv

data.to_csv(
    os.path.join(PROCESSED_PATH, "literacy_2025_processed.csv"),
    index=False
)

In [11]:
datasets = {
    "schools": schools,
    "eys": eys,
    "reading_fondness": reading_fondness,
    "completion": completion,
    "libraries": libraries,
    "facilities": facilities,
    "literacy": literacy,
    "education_25plus": education_25plus
}

In [16]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns)


schools
Shape: (42, 10)
Columns:
Index(['Provinsi', 'Number of Schools in Primary Schools (Public)',
       'Number of Schools in Primary Schools (Private)',
       'Number of Schools in Primary Schools (Public + Private)',
       'Number of Teachers in Primary Schools (Public)',
       'Number of Teachers in Primary Schools (Private)',
       'Number of Teachers in Primary Schools (Public + Private)',
       'Number of Pupils in Primary Schools (Public)',
       'Number of Pupils in Primary Schools (Private)',
       'Number of Pupils in Primary Schools (Public + Private)'],
      dtype='object')

eys
Shape: (579, 3)
Columns:
Index(['region', 'eys_male', 'eys_female'], dtype='object')

reading_fondness
Shape: (39, 5)
Columns:
Index(['Provinsi', 'Level of Reading Fondness', 'TGM-Pre Reading',
       'TGM-Reading', 'TGM-Post Reading'],
      dtype='object')

completion
Shape: (39, 4)
Columns:
MultiIndex([(             'Unnamed: 0_level_0', 'Unnamed: 0_level_1'),
            ( 'Elementa